# 1er Workshop Eje de Innovacion — Small Open Language Models

**Red Feminista de IA**

---

### Pre-requisitos
- Python >= 3.12
- `ipykernel` instalado (para ejecutar este notebook)

**Todo lo demas se instala automaticamente en la primera celda de codigo.**

### Modelos del workshop
- **Default:** `Qwen/Qwen2.5-1.5B-Instruct`
- **Fallback (hardware limitado):** `Qwen/Qwen2.5-0.5B-Instruct`

## 0. Pathways segun nivel y hardware

### 0.1 Objetivo final
Aplicacion del contenido explorado a un caso de uso de cada proyecto.

### 0.2 Elegi tu camino

| Pathway | Secciones | Descripcion |
|---------|-----------|-------------|
| 🟢 Exploracion simple | 1 → 3 → 6 → 8 → 12 | Setup → inferencia basica → Ollama → evaluacion → aplicacion |
| 🟡 Experimentacion | 1 → 3 → 4 → 5 → 8 → 12 | Setup → inferencia → eficiencia → tokenizacion → evaluacion → aplicacion |
| 🔴 Profundidad | 1 → 3 → 4 → 5 → 8 → 9 → 10 → 11 → 12 | Flujo completo con fine-tuning |
| ⚙️ Hardware limitado | 1 → 6 → 7 → 8 → 12 | Setup → Ollama → llama.cpp → evaluacion → aplicacion |

### 0.3 Elegi segun tu hardware
- 🖥️ **Solo CPU** → Priorizar secciones 6 (Ollama) y 7 (llama.cpp)
- ⚡ **GPU disponible** → Flujo completo con Hugging Face (3 → 10)
- ❓ **No sabes que tenes** → Avanzar hasta 1.4 (deteccion de hardware)

### 0.4 Como usar este notebook
- ▶️ **Ejecutar celda** → base funcional inmediata
- 🧪 **Proba esto** → modificar y experimentar
- 🚀 **Ir mas alla** → profundizacion opcional

---
## 1. Setup local (adaptado a hardware)

### 1.1-1.2 Entorno e instalacion base
La siguiente celda instala todas las dependencias necesarias.

In [ ]:
import subprocess
import sys

# Asegurar que pip este disponible
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "--version"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
except subprocess.CalledProcessError:
    print("⏳ Instalando pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--default-pip"])

paquetes = [
    "torch",
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "trl",
    "evaluate",
    "scikit-learn",
    "sentencepiece",
    "protobuf",
    "requests",
    "huggingface_hub",
]

print("📦 Instalando paquetes necesarios (puede tardar unos minutos)...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + paquetes
)
print("✅ Paquetes instalados correctamente")

In [ ]:
# ═══ Imports y configuracion ═══
import torch
import gc
import time
import json
import random
import os
import warnings
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# ═══ Configuracion del modelo ═══
NOMBRE_MODELO_DEFAULT = "Qwen/Qwen2.5-1.5B-Instruct"
NOMBRE_MODELO_FALLBACK = "Qwen/Qwen2.5-0.5B-Instruct"

# Elegi tu modelo segun tu hardware:
# - NOMBRE_MODELO_DEFAULT: requiere ~16GB RAM (recomendado con GPU o >=16GB RAM)
# - NOMBRE_MODELO_FALLBACK: requiere ~8GB RAM (recomendado para CPU / hardware limitado)
nombre_modelo = NOMBRE_MODELO_FALLBACK

print(f"📌 Modelo seleccionado: {nombre_modelo}")
print(f"   (Cambiar a {NOMBRE_MODELO_DEFAULT} si tenes >=16GB RAM libres)")

In [ ]:
# ═══ 1.3 Deteccion de hardware ═══
import platform

print("=" * 55)
print("🔍 DETECCION DE HARDWARE")
print("=" * 55)

print(f"\n💻 Sistema: {platform.system()} {platform.machine()}")
print(f"   Python: {sys.version.split()[0]}")

# RAM
if platform.system() == "Darwin":
    ram_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
elif platform.system() == "Linux":
    ram_bytes = 0
    with open("/proc/meminfo") as f:
        for linea in f:
            if "MemTotal" in linea:
                ram_bytes = int(linea.split()[1]) * 1024
                break
else:
    ram_bytes = 0

ram_gb = ram_bytes / (1024**3) if ram_bytes else 0
if ram_gb > 0:
    print(f"   RAM total: {ram_gb:.1f} GB")

# GPU / MPS
hay_cuda = torch.cuda.is_available()
hay_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

if hay_cuda:
    nombre_gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"\n⚡ GPU detectada: {nombre_gpu} ({vram_gb:.1f} GB VRAM)")
elif hay_mps:
    print(f"\n⚡ GPU detectada: 🍎 Apple Silicon (MPS backend)")
else:
    print(f"\n🖥️ Solo CPU disponible")

# Recomendacion
print("\n" + "=" * 55)
if hay_cuda or hay_mps:
    print("✅ Recomendacion: Flujo completo con Hugging Face (pathway 🔴)")
elif ram_gb >= 8:
    print("✅ Recomendacion: Podes usar Hugging Face en CPU (pathway 🟡)")
    print("   Tambien podes usar Ollama/llama.cpp para mas velocidad")
else:
    print("⚠️ Recomendacion: Usar Ollama o llama.cpp (pathway ⚙️)")
print("=" * 55)

In [ ]:
# ═══ 1.4 Setup alternativo: Ollama y llama.cpp ═══

import platform
import shutil

sistema = platform.system()  # 'Darwin', 'Linux', 'Windows'
print(f"🖥️  Sistema operativo detectado: {sistema}\n")

# ── Ollama ────────────────────────────────────────────────
print("📦 Verificando Ollama...")
if shutil.which("ollama") is not None:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print(f"   ✅ Ollama ya instalado: {result.stdout.strip()}")
else:
    print("   ⏳ Instalando Ollama...")
    try:
        if sistema == "Darwin":
            subprocess.check_call(["brew", "install", "ollama"])
        elif sistema == "Linux":
            subprocess.check_call(
                "curl -fsSL https://ollama.ai/install.sh | sh",
                shell=True,
            )
        elif sistema == "Windows":
            subprocess.check_call(
                ["winget", "install", "--id", "Ollama.Ollama", "-e", "--silent"]
            )
        else:
            print(f"   ⚠️  Sistema '{sistema}' no reconocido.")
            print("      Instalar manualmente desde: https://ollama.ai/download")
        if shutil.which("ollama") is not None:
            print("   ✅ Ollama instalado correctamente.")
    except FileNotFoundError as e:
        print(f"   ⚠️  Comando no encontrado: {e}")
        print("      Instalar manualmente desde: https://ollama.ai/download")
    except subprocess.CalledProcessError as e:
        print(f"   ❌ Error durante la instalacion de Ollama: {e}")

# ── llama-cpp-python ──────────────────────────────────────
print("\n📦 Instalando llama-cpp-python...")
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"]
    )
    print("   ✅ llama-cpp-python instalado correctamente.")
    print("   ℹ️  Requiere compilador C++ (cmake/clang). En macOS suele funcionar sin configuracion extra.")
except subprocess.CalledProcessError as e:
    print(f"   ❌ Error instalando llama-cpp-python: {e}")
    print("      En Linux: sudo apt install cmake build-essential")
    print("      En Windows: instalar Visual Studio Build Tools")

---
## 2. Small Open Language Models (contexto minimo)

### 2.1 Que significa "small"
Un **Small Language Model (SLM)** tiene tipicamente entre **0.5B y 7B parametros**, en contraste con los LLMs como GPT-4 (estimado >1T parametros) o LLaMA-70B.

Estos modelos son lo suficientemente capaces para muchas tareas y lo suficientemente livianos para correr **localmente** en hardware accesible.

### 2.2 Tradeoffs reales

| Aspecto | SLM (1-3B) | LLM (70B+) |
|---------|-----------|------------|
| RAM necesaria (inferencia) | 2-6 GB | >140 GB |
| Velocidad (CPU) | Segundos | Minutos |
| Calidad general | Buena para tareas especificas | Excelente generalista |
| Costo de deploy | Bajo / gratis | Alto (GPU necesaria) |
| Privacidad | Total (local) | No garantizada (proveedor) |
| Fine-tuning | Factible en laptop | Requiere cluster |

### 2.3 Casos de uso aplicados
- **Clasificacion de textos**
- **Resumen automatico**
- **Extraccion de informacion**
- **Asistentes especializados**
- **Traduccion / adaptacion linguistica**

> 🧪 **Proba esto:** Pensa en tu proyecto — ¿que tarea especifica podria hacer un SLM?

---
## 3. Inferencia base con Hugging Face

### 3.1 Cargar modelo
Hay dos formas principales:
- **Pipeline:** interfaz de alto nivel, rapida de usar
- **Manual:** mas control sobre tokenizacion y generacion

In [ ]:
# ═══ 3.1 Cargar modelo y tokenizador ═══
print(f"⏳ Cargando modelo: {nombre_modelo}")
print("   (la primera vez descarga los pesos, puede tardar varios minutos)")

tokenizador = AutoTokenizer.from_pretrained(nombre_modelo)
if tokenizador.pad_token is None:
    tokenizador.pad_token = tokenizador.eos_token

try:
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre_modelo, torch_dtype="auto", device_map="auto"
    )
except Exception as error_carga:
    print(f"⚠️ device_map='auto' fallo ({error_carga}), cargando en CPU...")
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre_modelo, torch_dtype=torch.float32
    )

dispositivo = next(modelo.parameters()).device
print(f"\n✅ Modelo cargado en: {dispositivo}")
print(f"   Parametros: {sum(p.numel() for p in modelo.parameters()) / 1e6:.0f}M")
print(f"   Dtype: {next(modelo.parameters()).dtype}")

In [ ]:
# ═══ 3.1b Inferencia con pipeline ═══
generador = pipeline("text-generation", model=modelo, tokenizer=tokenizador)

mensajes = [{"role": "user", "content": "Explica brevemente que es una inteligencia artifical"}]
respuesta = generador(mensajes)

print("🤖 Respuesta (pipeline):")
print(respuesta[0]["generated_text"][-1]["content"])

In [ ]:
# ═══ 3.2 Generacion manual con distintos parametros ═══

def generar_respuesta(prompt, max_tokens=100, temperatura=0.0, modelo_usar=None):
    """Genera una respuesta dado un prompt o una lista de prompts (batching)."""
    _modelo = modelo_usar if modelo_usar is not None else modelo
    _dispositivo = next(_modelo.parameters()).device

    # Soporte para un prompt individual o una lista (batching)
    es_lista = isinstance(prompt, list)
    prompts = prompt if es_lista else [prompt]

    # Construir lista de conversaciones (lista de listas de mensajes)
    conversaciones = [[{"role": "user", "content": p}] for p in prompts]

    # apply_chat_template con tokenize=True aplica la plantilla y tokeniza en un solo paso
    # Los modelos causales requieren padding a la izquierda
    entrada = tokenizador.apply_chat_template(
        conversaciones,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        padding=True,
        return_dict=True,
    )
    entrada = {k: v.to(_dispositivo) for k, v in entrada.items()}

    params_gen = {
        "max_new_tokens": max_tokens,
        "pad_token_id": tokenizador.pad_token_id,
    }
    if temperatura > 0:
        params_gen["do_sample"] = True
        params_gen["temperature"] = temperatura
    else:
        params_gen["do_sample"] = False

    with torch.no_grad():
        salida = _modelo.generate(**entrada, **params_gen)

    # Decodificar solo los tokens nuevos (sin el prompt de entrada) usando batch_decode
    input_len = entrada["input_ids"].shape[1]
    respuestas = tokenizador.batch_decode(salida[:, input_len:], skip_special_tokens=True)

    return respuestas if es_lista else respuestas[0]


# Probar con diferentes temperaturas
prompt_ejemplo = "Crea un poema sobre las abejas:"

configuraciones = {
    "Determinista (temp=0)": 0.0,
    "Determinista (temp=0) - segundo intento, debería ser idéntico al primero": 0.0,
    "Conservadora (temp=0.3)": 0.3,
    "Balanceada (temp=0.7)": 0.7,
    "Creativa (temp=1.2)": 1.2,
}

for nombre_config, temp in configuraciones.items():
    print(f"\n{'='*60}")
    print(f"📝 {nombre_config}")
    print(f"{'='*60}")
    respuesta = generar_respuesta(prompt_ejemplo, max_tokens=80, temperatura=temp)
    print(respuesta[:300])

### 3.3 Primer ejercicio
🧪 **Proba esto:** Modifica el prompt para que se relacione con tu proyecto.

In [ ]:
# ═══ 3.3 Ejercicio: tu primer prompt aplicado ═══

# 🧪 Modifica este prompt para tu proyecto:
mi_prompt = "¿Cuales son las principales aplicaciones de la inteligencia artificial en la vida cotidiana?"

respuesta = generar_respuesta(mi_prompt, max_tokens=150)
print("🤖 Respuesta:")
print(respuesta)

In [ ]:
# ═══ 3.4 Ir mas alla: batching y streaming ═══

# --- Batching: procesar multiples prompts en una sola inferencia ---
prompts_batch = [
    "¿Que es el aprendizaje automatico?",
    "¿Cuales son los tipos de redes neuronales?",
    "¿Que son los datos abiertos?",
]

print("📦 Procesando batch de prompts:\n")
respuestas_det = generar_respuesta(prompts_batch, max_tokens=60)
for i, (prompt, respuesta) in enumerate(zip(prompts_batch, respuestas_det)):
    print(f"📝 Prompt {i+1}: {prompt}")
    print(f"🤖 {respuesta[:200]}")
    print()

# --- Streaming ---
from transformers import TextStreamer

print("\n--- Streaming (la respuesta aparece token por token) ---\n")
streamer = TextStreamer(tokenizador, skip_special_tokens=True, skip_prompt=True)

mensajes = [{"role": "user", "content": "Explicame brevemente que es machine learning."}]
texto_chat = tokenizador.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
entrada = tokenizador(texto_chat, return_tensors="pt")
_dispositivo = next(modelo.parameters()).device
entrada = {k: v.to(_dispositivo) for k, v in entrada.items()}

print("🤖 Respuesta (streaming):")
with torch.no_grad():
    modelo.generate(
        **entrada, max_new_tokens=80,
        streamer=streamer, pad_token_id=tokenizador.pad_token_id,
    )

---
## 4. Inferencia eficiente (Hugging Face)

### 4.1 Cuantizacion (8-bit / 4-bit)
La **cuantizacion** reduce la precision numerica de los pesos del modelo para ahorrar memoria y acelerar la inferencia.
La reducción no es lineal porque solo se comprimen las capas lineales, mientras que las capas de embedding, normalización y cabezales (heads) permanecen en alta precisión (16/32-bit) para evitar la pérdida de inteligencia del modelo.

| Precision | Bytes/param | RAM (0.5B) | Calidad |
|-----------|------------|-----------|---------|
| float32 | 4 | ~1.84 GB | Maxima |
| float16/bf16 | 2 | ~0.92 GB | Muy buena |
| 8-bit (int8) | 1 | ~0.59 GB | Buena |
| 4-bit (nf4) | 0.5 | ~0.42 GB | Aceptable |

> ⚠️ La cuantizacion con `bitsandbytes` requiere GPU NVIDIA (CUDA). En CPU, las alternativas son Ollama y llama.cpp (secciones 6-7) que usan formato GGUF.

In [ ]:
# ═══ 4.1-4.2 Cuantizacion y uso de memoria ═══

prompt_prueba = "¿Que es la inteligencia artificial? Responde en una oracion."

def memoria_modelo(m):
    return m.get_memory_footprint() / (1024**3)

def probar_precision(etiqueta, m):
    dtype = next(m.parameters()).dtype
    mem = memoria_modelo(m)
    resp = generar_respuesta(prompt_prueba, max_tokens=50, modelo_usar=m)
    print(f"\n{'='*58}")
    print(f"🔢 {etiqueta}")
    print(f"   Dtype:     {dtype}")
    print(f"   Memoria:   {mem:.2f} GB")
    print(f"   Respuesta: {resp[:150]}")
    return mem

resultados_precision = {}

# ── float32 (precision completa) ─────────────────────────
print("⏳ Cargando modelo en float32 (precision completa)...")
modelo_fp32 = AutoModelForCausalLM.from_pretrained(
    nombre_modelo, torch_dtype=torch.float32, device_map="auto"
)
resultados_precision["float32"] = probar_precision("float32 (precision completa)", modelo_fp32)
del modelo_fp32
gc.collect()

# ── float16 (16-bit) ─────────────────────────────────────
print("\n⏳ Cargando modelo en float16 (16-bit)...")
modelo_fp16 = AutoModelForCausalLM.from_pretrained(
    nombre_modelo, torch_dtype=torch.float16, device_map="auto"
)
resultados_precision["float16"] = probar_precision("float16 (16-bit)", modelo_fp16)
del modelo_fp16
gc.collect()

# ── int8 y nf4 (requieren GPU CUDA + bitsandbytes) ───────
if hay_cuda or hay_mps:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
        from transformers import BitsAndBytesConfig
        torch.cuda.empty_cache()

        # 8-bit (int8)
        print("\n⏳ Cargando modelo en 8-bit (int8)...")
        modelo_8bit = AutoModelForCausalLM.from_pretrained(
            nombre_modelo,
            quantization_config=BitsAndBytesConfig(load_in_8bit=True),
            device_map="auto",
        )
        resultados_precision["int8"] = probar_precision("int8 (8-bit)", modelo_8bit)
        del modelo_8bit
        gc.collect()
        torch.cuda.empty_cache()

        # 4-bit (nf4)
        print("\n⏳ Cargando modelo en 4-bit (nf4)...")
        modelo_4bit = AutoModelForCausalLM.from_pretrained(
            nombre_modelo,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            ),
            device_map="auto",
        )
        resultados_precision["nf4"] = probar_precision("nf4 (4-bit)", modelo_4bit)
        del modelo_4bit
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"\n⚠️ Error con cuantizacion bitsandbytes: {e}")
else:
    print("\n⚠️ Cuantizacion 8-bit y 4-bit requieren GPU NVIDIA (CUDA).")
    print("   En CPU/MPS solo estan disponibles float32 y float16.")
    print("   Alternativas para CPU: Ollama y llama.cpp (secciones 6-7).")

# ── Resumen comparativo ───────────────────────────────────
if resultados_precision:
    mem_base = resultados_precision.get("float32", next(iter(resultados_precision.values())))
    print(f"\n{'='*58}")
    print("📊 Resumen comparativo de memoria:")
    print(f"{'Precision':<12}  {'Memoria (GB)':>12}  {'Reduccion':>10}")
    print("-" * 38)
    for nombre, mem in resultados_precision.items():
        reduccion = (1 - mem / mem_base) * 100
        print(f"{nombre:<12}  {mem:>12.2f}  {reduccion:>9.0f}%")

In [ ]:
# ═══ 4.3-4.4 Velocidad de inferencia ═══

prompt_velocidad = "Explica brevemente que es deep learning."
n_repeticiones = 10
max_tokens_velocidad = 50

def medir_velocidad(m, etiqueta):
    tiempos = []
    for i in range(n_repeticiones):
        inicio = time.time()
        _ = generar_respuesta(prompt_velocidad, max_tokens=max_tokens_velocidad, modelo_usar=m)
        tiempos.append(time.time() - inicio)
    promedio = np.mean(tiempos)
    tps = max_tokens_velocidad / promedio
    dispositivo = next(m.parameters()).device
    dtype = next(m.parameters()).dtype
    print(f"\n{'='*52}")
    print(f"⚡ {etiqueta}")
    print(f"   Dispositivo: {dispositivo}  |  Dtype: {dtype}")
    print(f"   Tiempo promedio: {promedio:.2f}s (±{np.std(tiempos):.2f}s)")
    print(f"   Tokens/segundo estimado: ~{tps:.1f}")
    return promedio, tps

resultados_velocidad = {}

# ── Dispositivo actual (GPU/MPS si esta disponible) ───────
print("⏱️ Midiendo velocidad de inferencia...\n")
disp_actual = str(next(modelo.parameters()).device).split(":")[0]
t_acelerador, tps_acelerador = medir_velocidad(modelo, f"Acelerador ({disp_actual})")
resultados_velocidad[disp_actual] = (t_acelerador, tps_acelerador)

# ── Comparacion con CPU (solo si hay GPU o MPS) ───────────
if hay_cuda or hay_mps:
    print("\n⏳ Cargando modelo en CPU para comparacion...")
    modelo_cpu = AutoModelForCausalLM.from_pretrained(
        nombre_modelo,
        torch_dtype=next(modelo.parameters()).dtype,
        device_map="cpu",
    )
    t_cpu, tps_cpu = medir_velocidad(modelo_cpu, "CPU")
    resultados_velocidad["cpu"] = (t_cpu, tps_cpu)
    del modelo_cpu
    gc.collect()

    print(f"\n{'='*52}")
    print("📊 Resumen comparativo:")
    print(f"{'Dispositivo':<14}  {'Promedio (s)':>13}  {'Tok/s':>7}  {'Aceleracion':>12}")
    print("-" * 52)
    for disp, (t, tps) in resultados_velocidad.items():
        aceleracion = t_cpu / t if disp != "cpu" else 1.0
        print(f"{disp:<14}  {t:>13.2f}  {tps:>7.1f}  {aceleracion:>11.1f}x")
else:
    print(f"\n{'='*52}")
    print("📊 Solo CPU disponible — sin comparacion de aceleracion.")

---
## 5. Tokenizacion, longitud y fertility rate

### 5.1 Que es un token
Un **token** es la unidad minima que procesa un modelo de lenguaje. No es necesariamente una palabra: puede ser una subpalabra, un caracter, o incluso un byte.

**¿Por que importa?**
- El **costo** de usar un modelo se mide en tokens
- La **ventana de contexto** tiene un limite fijo de tokens
- Un mismo texto en distintos idiomas puede tener cantidades muy diferentes de tokens

In [ ]:
# ═══ 5.1-5.2 Como tokeniza el modelo ═══

texto_ejemplo = "La inteligencia artificial puede transformar la sociedad"

tokens_ids = tokenizador.encode(texto_ejemplo)
tokens_texto = tokenizador.tokenize(texto_ejemplo)

print(f"📝 Texto: '{texto_ejemplo}'")
print(f"   Palabras: {len(texto_ejemplo.split())}")
print(f"   Tokens: {len(tokens_ids)}")
print(f"   Tokens (texto): {tokens_texto}")
print()

# Decodificar token por token
print("🔍 Decodificacion token por token:")
for tid in tokens_ids:
    fragmento = tokenizador.decode([tid])
    print(f"   ID {tid:6d} → '{fragmento}'")

In [ ]:
# ═══ 5.2-5.3 Tokenizacion desigual entre idiomas ═══

frases = {
    "Espanol":   "La inteligencia artificial puede transformar la sociedad",
    "Ingles":    "Artificial intelligence can transform society",
    "Portugues": "A inteligencia artificial pode transformar a sociedade",
    "Frances":   "L'intelligence artificielle peut transformer la societe",
    "Quechua":   "Kapchisqa yachayqa llaqtata tikranman",
    "Guarani":   "Ava japopyre arandu ikatu omoambue avano'õme",
    "Arabe":     "يمكن للذكاء الاصطناعي أن يغير المجتمع",
    "Hebreo":    "בינה מלאכותית יכולה לשנות את החברה",
}

print("📊 Comparacion de tokenizacion entre idiomas\n")
print(f"{'Idioma':<12} {'Palabras':<10} {'Tokens':<10} {'Ratio':<10}")
print("-" * 42)

for idioma, frase in frases.items():
    tokens = tokenizador.encode(frase)
    palabras = len(frase.split())
    ratio = len(tokens) / max(palabras, 1)
    print(f"{idioma:<12} {palabras:<10} {len(tokens):<10} {ratio:<10.2f}")

print()
print("💡 Un ratio mas alto significa que el modelo necesita mas tokens para")
print("   representar el mismo contenido → mayor costo y menor contexto disponible.")

In [ ]:
# ═══ 5.4-5.5 Fertility Rate ═══

print("📊 Fertility Rate: tokens por unidad linguistica\n")

textos_comparacion = {
    "Espanol": [
        "La inteligencia artificial transforma la medicina",
        "Los datos abiertos permiten transparencia",
        "La tecnologia debe ser accesible para todos",
    ],
    "Ingles": [
        "Artificial intelligence transforms medicine today",
        "Open data enables transparency always",
        "Technology must be accessible for everyone",
    ],
    "Quechua": [
        "Yachay rurana atinmi hampiq llamkayta tikray",
        "Kichqa willakuykunaqa sutichasqa kanan",
        "Tecnologia lliw runakunapaqmi kanan",
    ],
}

resultados_fr = {}
for idioma, textos in textos_comparacion.items():
    fertility_rates = []
    for texto in textos:
        tokens = tokenizador.encode(texto)
        palabras = texto.split()
        fr = len(tokens) / max(len(palabras), 1)
        fertility_rates.append(fr)

    promedio_fr = np.mean(fertility_rates)
    resultados_fr[idioma] = promedio_fr
    print(f"  {idioma}: fertility rate promedio = {promedio_fr:.2f} tokens/palabra")
    for i, texto in enumerate(textos):
        toks = len(tokenizador.encode(texto))
        pals = len(texto.split())
        print(f"    '{texto[:50]}...' → {pals} palabras, {toks} tokens")
    print()

print("💡 Idiomas con menor representacion en los datos de entrenamiento")
print("   tienden a tener fertility rates mas altos, lo que implica:")
print("   • Mayor costo computacional por el mismo contenido")
print("   • Menor ventana de contexto efectiva")
print("   • Potencial perdida de calidad en las respuestas")

### 5.6 Discusion
- ¿Que implicancias tiene la tokenizacion desigual para distintos idiomas?
- Si un idioma necesita 3x mas tokens para decir lo mismo, ¿que pasa con el costo y la calidad?
- ¿Como afecta esto a lenguas indigenas o variedades linguisticas menos representadas?

> 🧪 **Proba esto:** Agrega frases en otros idiomas o dialectos y observa la fragmentacion.

---
## 6. Inferencia local simple con Ollama

**Ollama** simplifica correr modelos localmente. Es ideal para:
- Prototipado rapido sin escribir codigo Python complejo
- Correr modelos cuantizados (GGUF) eficientemente en CPU
- APIs locales compatibles con OpenAI

### 6.1 Setup minimo manual (no hace falta correrlo, se corre automaticamente en la siguiente celda)
```bash
ollama serve           # iniciar servidor
ollama pull hf.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q2_K  # descargar modelo
ollama run hf.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q2_K   # chat interactivo
```

In [ ]:
# ═══ 6.1 Setup de Ollama ═══

import shutil
import time

# ── 1. Verificar instalacion ──────────────────────────────
if shutil.which("ollama") is None:
    print("⚠️  Ollama no encontrado. Ejecuta primero la celda 1.4 para instalarlo.")
else:
    print(f"✅ Ollama encontrado en: {shutil.which('ollama')}")

    # ── 2. Iniciar servidor en segundo plano (si no corre ya) ─
    # Forzar uso de CPU en Ollama (ignorar GPU/MPS aunque esten disponibles)
    import os
    os.environ["OLLAMA_NUM_GPU"] = "0"
    try:
        r = __import__("requests").get("http://localhost:11434/api/tags", timeout=2)
        print("✅ Servidor Ollama ya en ejecucion.")
        print("   ℹ️  Si el servidor ya estaba corriendo, reinicialo para aplicar CPU-only.")
    except Exception:
        print("⏳ Iniciando servidor Ollama en segundo plano (CPU-only)...")
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            env=os.environ,
        )
        # Esperar a que levante
        for _ in range(10):
            time.sleep(1)
            try:
                __import__("requests").get("http://localhost:11434/api/tags", timeout=1)
                print("✅ Servidor Ollama iniciado.")
                break
            except Exception:
                pass
        else:
            print("❌ El servidor no respondio a tiempo. Verifica con: ollama serve")

    # ── 3. Descargar modelo si no esta disponible ─────────────
    MODELO_OLLAMA = "hf.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q4_K_M"
    print(f"\n⏳ Verificando modelo '{MODELO_OLLAMA}'...")
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True, text=True, check=True,
        )
        if MODELO_OLLAMA in result.stdout:
            print(f"   ✅ Modelo '{MODELO_OLLAMA}' ya disponible.")
        else:
            print(f"   ⏳ Descargando '{MODELO_OLLAMA}' (puede tardar unos minutos)...")
            subprocess.check_call(["ollama", "pull", MODELO_OLLAMA])
        print(f"   ✅ Modelo '{MODELO_OLLAMA}' descargado.")
    except subprocess.CalledProcessError as e:
        print(f"   ❌ Error: {e}")


In [ ]:
# ═══ 6.2-6.3 Uso de Ollama desde Python ═══
import requests

OLLAMA_URL = "http://localhost:11434"
ollama_disponible = False

try:
    r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
    if r.status_code == 200:
        ollama_disponible = True
        modelos_ollama = r.json().get("models", [])
        print("✅ Ollama disponible. Modelos instalados:")
        for m in modelos_ollama:
            print(f"   - {m['name']}")
except Exception:
    pass

if not ollama_disponible:
    print("ℹ️  Ollama no esta disponible en este entorno.")
    print("   Para instalar:")
    print("   macOS: brew install ollama")
    print("   Linux: curl -fsSL https://ollama.ai/install.sh | sh")
    print("   Luego: ollama serve && ollama pull qwen2.5:0.5b")

In [ ]:
# ═══ 6.4-6.5 Ejercicio con Ollama ═══

if ollama_disponible:
    prompt_ollama = "¿Que es la inteligencia artificial?"

    print(f"📝 Prompt: {prompt_ollama}")
    print(f"🔧 Modelo: {MODELO_OLLAMA}\n")

    try:
        respuesta_ollama = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": MODELO_OLLAMA,
                "prompt": prompt_ollama,
                "stream": False,
                "options": {"num_gpu": 0},
            },
            timeout=120,
        )
        if respuesta_ollama.status_code == 200:
            texto_ollama = respuesta_ollama.json()["response"]
            print(f"🤖 Respuesta de Ollama:\n{texto_ollama}")
        else:
            print(f"⚠️ Error: {respuesta_ollama.status_code}")
            print(f"   Asegurate de tener el modelo descargado: ollama pull {MODELO_OLLAMA}")
    except Exception as e:
        print(f"⚠️ Error al conectar con Ollama: {e}")

    # Cuando usar Ollama vs Hugging Face:
    print("\n📋 Ollama vs Hugging Face:")
    print("   Ollama: mas simple, modelos cuantizados, ideal para CPU")
    print("   HF: mas control, fine-tuning, mejor para experimentacion")

    # ── Comparacion de velocidad HF vs Ollama ─────────────────
    try:
        _ = modelo  # verificar que el modelo HF esta cargado
        hf_disponible = True
    except NameError:
        hf_disponible = False

    if hf_disponible:
        N_RUNS = 10
        MAX_TOK = 50
        print(f"\n{'='*58}")
        print(f"⚡ Comparacion de velocidad HF vs Ollama ({N_RUNS} ejecuciones)")
        print(f"   Prompt: {prompt_ollama}")
        print(f"   Max tokens: {MAX_TOK}")

        # Medir HF — contar tokens reales del output
        tiempos_hf = []
        tokens_hf = []
        for i in range(N_RUNS):
            t0 = time.time()
            resp_hf = generar_respuesta(prompt_ollama, max_tokens=MAX_TOK)
            tiempos_hf.append(time.time() - t0)
            tokens_hf.append(len(tokenizador.encode(resp_hf)))
        promedio_hf = np.mean(tiempos_hf)
        tps_hf = float(np.mean([tok / t for tok, t in zip(tokens_hf, tiempos_hf)]))

        print(f"\n🤗 Hugging Face  ({next(modelo.parameters()).device}, {next(modelo.parameters()).dtype})")
        for i, (t, tok) in enumerate(zip(tiempos_hf, tokens_hf), 1):
            print(f"   Run {i}: {t:.2f}s  ({tok} tokens)")
        print(f"   Promedio: {promedio_hf:.2f}s  |  ~{tps_hf:.1f} tok/s")

        # Medir Ollama — usar eval_count del JSON de respuesta
        tiempos_ollama = []
        tokens_ollama = []
        for i in range(N_RUNS):
            t0 = time.time()
            r_t = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json={
                    "model": MODELO_OLLAMA,
                    "prompt": prompt_ollama,
                    "stream": False,
                    "options": {"num_predict": MAX_TOK, "num_gpu": 0},
                },
                timeout=120,
            )
            tiempos_ollama.append(time.time() - t0)
            tokens_ollama.append(r_t.json().get("eval_count", MAX_TOK))
        promedio_ollama = np.mean(tiempos_ollama)
        tps_ollama = float(np.mean([tok / t for tok, t in zip(tokens_ollama, tiempos_ollama)]))

        print(f"\n🦙 Ollama  ({MODELO_OLLAMA})")
        for i, (t, tok) in enumerate(zip(tiempos_ollama, tokens_ollama), 1):
            print(f"   Run {i}: {t:.2f}s  ({tok} tokens)")
        print(f"   Promedio: {promedio_ollama:.2f}s  |  ~{tps_ollama:.1f} tok/s")

        # Resumen
        mas_rapido = "HF" if promedio_hf < promedio_ollama else "Ollama"
        ratio_velocidad = max(promedio_hf, promedio_ollama) / min(promedio_hf, promedio_ollama)
        print(f"\n{'='*58}")
        print(f"{'Motor':<12}  {'Promedio (s)':>13}  {'Tok/s':>7}")
        print("-" * 36)
        print(f"{'HF':<12}  {promedio_hf:>13.2f}  {tps_hf:>7.1f}")
        print(f"{'Ollama':<12}  {promedio_ollama:>13.2f}  {tps_ollama:>7.1f}")
        print(f"\n🏆 {mas_rapido} es {ratio_velocidad:.1f}x mas rapido en este entorno.")
    else:
        print("\nℹ️  Modelo HF no cargado — sin comparacion de velocidad.")
        print("   Esta seccion requiere tener Ollama corriendo localmente")

else:
    print("⏩ Saltando (Ollama no disponible)")


---
## 7. Inferencia optimizada con llama.cpp

### 7.1 Que es GGUF y cuantizacion
**GGUF** es un formato de modelo optimizado para correr en CPU. Almacena los pesos cuantizados de forma eficiente.

**llama.cpp** es una implementacion en C++ que ejecuta modelos GGUF con alta eficiencia, usando SIMD, multithreading y otros trucos de bajo nivel.

### 7.3 Parametros de performance
- `n_threads`: numero de hilos de CPU a usar
- `n_batch`: tamano del batch de tokens
- `n_ctx`: tamano de la ventana de contexto

In [ ]:
# ═══ 7.2-7.5 Inferencia con llama.cpp via Python ═══

llama_cpp_disponible = False
try:
    from llama_cpp import Llama
    llama_cpp_disponible = True
    print("✅ llama-cpp-python disponible")
except ImportError:
    print("ℹ️  llama-cpp-python no esta instalado.")
    print("   Para instalar: pip install llama-cpp-python")
    print("   Nota: requiere compilador C++ (cmake, clang)")

if llama_cpp_disponible:
    # Repo HF equivalente y nombre de archivo
    GGUF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
    GGUF_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"

    from huggingface_hub import hf_hub_download
    print(f"\n⏳ Descargando modelo GGUF '{GGUF_FILE}' desde HuggingFace...")
    ruta_gguf = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE)
    print(f"   ✅ Guardado en: {ruta_gguf}")

    # ── Cargar con llama.cpp — CPU only (n_gpu_layers=0) ──────
    n_threads = os.cpu_count()
    print(f"\n⏳ Cargando modelo en llama.cpp (CPU only, {n_threads} hilos)...")
    llm = Llama(
        model_path=ruta_gguf,
        n_ctx=512,
        n_threads=n_threads,
        n_gpu_layers=0,       # forzar CPU aunque haya GPU/MPS
        verbose=False,
    )
    print("   ✅ Modelo cargado.")

    # ── Inferencia de prueba ───────────────────────────────────
    prompt_llama = prompt_ollama
    MAX_TOK_LLAMA = 50
    N_RUNS_LLAMA = 10

    print(f"\n📝 Prompt: {prompt_llama}")
    salida_llama = llm(prompt_llama, max_tokens=MAX_TOK_LLAMA, echo=False)
    print(f"🤖 Respuesta llama.cpp:\n{salida_llama['choices'][0]['text'].strip()}")

    # ── Benchmark llama.cpp — usar completion_tokens del output ─
    tiempos_llama = []
    tokens_llama = []
    for i in range(N_RUNS_LLAMA):
        t0 = time.time()
        out_llama = llm(prompt_llama, max_tokens=MAX_TOK_LLAMA, echo=False)
        tiempos_llama.append(time.time() - t0)
        tokens_llama.append(out_llama["usage"]["completion_tokens"])
    promedio_llama = np.mean(tiempos_llama)
    tps_llama = float(np.mean([tok / t for tok, t in zip(tokens_llama, tiempos_llama)]))

    print(f"\n⚡ llama.cpp ({N_RUNS_LLAMA} runs, CPU only):")
    for i, (t, tok) in enumerate(zip(tiempos_llama, tokens_llama), 1):
        print(f"   Run {i}: {t:.2f}s  ({tok} tokens)")
    print(f"   Promedio: {promedio_llama:.2f}s  |  ~{tps_llama:.1f} tok/s")

    # ── Comparacion con Ollama (si esta disponible) ───────────
    if ollama_disponible:
        print(f"\n🦙 Ollama  ({MODELO_OLLAMA}, CPU only):")
        tiempos_ollama_cmp = []
        tokens_ollama_cmp = []
        for i in range(N_RUNS_LLAMA):
            t0 = time.time()
            r_cmp = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json={
                    "model": MODELO_OLLAMA,
                    "prompt": prompt_llama,
                    "stream": False,
                    "options": {"num_predict": MAX_TOK_LLAMA, "num_gpu": 0},
                },
                timeout=120,
            )
            tiempos_ollama_cmp.append(time.time() - t0)
            tokens_ollama_cmp.append(r_cmp.json().get("eval_count", MAX_TOK_LLAMA))
        promedio_ollama_cmp = np.mean(tiempos_ollama_cmp)
        tps_ollama_cmp = float(np.mean([tok / t for tok, t in zip(tokens_ollama_cmp, tiempos_ollama_cmp)]))

        for i, (t, tok) in enumerate(zip(tiempos_ollama_cmp, tokens_ollama_cmp), 1):
            print(f"   Run {i}: {t:.2f}s  ({tok} tokens)")
        print(f"   Promedio: {promedio_ollama_cmp:.2f}s  |  ~{tps_ollama_cmp:.1f} tok/s")

        # Resumen
        ganador = "llama.cpp" if promedio_llama < promedio_ollama_cmp else "Ollama"
        factor = max(promedio_llama, promedio_ollama_cmp) / min(promedio_llama, promedio_ollama_cmp)
        print(f"\n{'='*58}")
        print(f"📊 Comparacion CPU — mismo modelo ({GGUF_FILE})")
        print(f"{'Motor':<14}  {'Promedio (s)':>13}  {'Tok/s':>7}")
        print("-" * 38)
        print(f"{'llama.cpp':<14}  {promedio_llama:>13.2f}  {tps_llama:>7.1f}")
        print(f"{'Ollama':<14}  {promedio_ollama_cmp:>13.2f}  {tps_ollama_cmp:>7.1f}")
        print(f"\n🏆 {ganador} es {factor:.2f}x mas rapido en este entorno.")
    else:
        print("\nℹ️  Ollama no disponible — comparacion omitida.")
        print(f"   llama.cpp: {promedio_llama:.2f}s promedio  |  ~{tps_llama:.1f} tok/s")

    del llm
    gc.collect()

else:
    print("\n⏩ llama-cpp-python no disponible — saltando seccion 7.")
